In [1]:
experiment = "ssp534-over"
import torch

In [2]:
from tensordict.tensordict import TensorDict

targets = []
for i in range(1, 2):
    target = TensorDict.load_memmap(
        f"/scratch/gclyne/memmap_filled_in/r{i}i1p1f1_{experiment}_interpolation.memmap"
    )
    # target_domain = (target['surface'] - train_dataset.data_mean['surface'] )/ train_dataset.data_std['surface']
    targets.append(target)
# abrupt = TensorDict.load_memmap('/lustre/fsn1/projects/rech/mlr/udy16au/memmap_filled_in_yearly/r1i1p1f1_abrupt-4xCO2_interpolation.memmap')
targets = torch.stack(targets)  # shape (ens, var, time, lat, lon)``

In [8]:
import pandas as pd
import xarray as xr

tas_np = targets[0]["surface"][0, :].view(-1, 12, 144, 144).mean(1).numpy()
# tas_np = targets[0]['surface'][0,:]
time = pd.date_range(start="2040-01-16T12:00:00", periods=61, freq="Y")
import numpy as np

lat = np.linspace(-89.375, 89.375, 144)
lon = np.linspace(0.0, 357.5, 144)
lat = np.linspace(-85.5, 85.5, 144)
lon = np.arange(0.0, 360.0, 360.0 / 144)

tas = xr.DataArray(
    tas_np,
    name="tas",
    dims=("time", "lat", "lon"),
    coords={
        "time": time,
        "lat": lat,
        "lon": lon,
        "height": 2.0,
    },
    attrs={
        "standard_name": "air_temperature",
        "long_name": "Near-Surface Air Temperature",
        "units": "K",
        "online_operation": "average",
        "cell_methods": "area: time: mean",
        "interval_operation": "900 s",
        "interval_write": "1 month",
        "description": "near-surface (usually, 2 meter) air temperature",
        "history": "none",
        "cell_measures": "area: areacella",
    },
)

/tmp/ipykernel_2421560/2440474570.py:5: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  time = pd.date_range(


In [9]:
tas.to_netcdf(
    f"tas_ann_IPSL-CM6A-LR_{experiment}_r1i1p1f1_g025.nc",
    encoding={
        "tas": {
            "zlib": True,
            "complevel": 4,
            "dtype": "float32",  # optional
        }
    },
)
# tas_mon_IPSL-CM6A-LR_ssp585_r2i1p1f1_g025